# CAVE

The CAVE nodes, wired the way references exist for.

*Exported from Coda v0.0.0-test on 2026-01-01.*

In [ ]:
# pip install caveclient numpy pandas sea-serpent

import os
import pandas as pd
import numpy as np
from caveclient import CAVEclient
import seaserpent as ss

In [ ]:
# Helpers, generated by Coda. These are the parts of the workflow that have
# no equivalent in the libraries this notebook imports, written out here so
# it stands on its own.

def coda_int64(values):
    """A column of ids as exact int64. Anything unreadable becomes 0."""

    def one(value):
        try:
            # Exact for an int and for decimal text of any width, which pd.to_numeric is
            # not: a single null in the column makes it answer float64, and an
            # eighteen-digit root id through a float is a different neuron.
            return int(value)
        except (TypeError, ValueError):
            try:
                return int(float(value))
            except (TypeError, ValueError):
                return 0

    return values.map(one).astype('int64')


def coda_annotation_columns(df, id_column):
    """Rename a CAVE table's columns to the two names Coda addresses by name."""
    renames = {}
    if id_column in df.columns:
        renames[id_column] = 'neuronId'
    for name in ('cell_type', 'celltype'):
        if name in df.columns and 'type' not in df.columns:
            renames[name] = 'type'
            break
    out = df.rename(columns=renames)
    if 'neuronId' not in out.columns:
        return out
    # A row with no id names no neuron, which is what Coda's shaping drops. An empty
    # string counts: SeaTable spells a blank cell that way.
    ids = out['neuronId']
    out = out[ids.notna() & (ids.astype(str) != '')].copy()
    ids = out['neuronId']
    if pd.api.types.is_float_dtype(ids):
        # Already lossy — a float64 cannot hold an eighteen-digit root id — but `str()` of
        # one is `7.2e+17`, which matches nothing at all. Integer text at least keeps the
        # shape of an id. Text and int columns go straight through, exact at any width.
        ids = coda_int64(ids)
    out['neuronId'] = ids.astype(str)
    return out


def coda_cave_neurons(
    client,
    neuron_table,
    id_column='pt_root_id',
    annotation_table=None,
    ref_column=None,
    system_column=None,
    value_column=None,
):
    """Coda's neuron index for a CAVE datastack: one row per neuron, a column per kind."""
    neurons = client.materialize.query_table(
        neuron_table, select_columns=['id', id_column], merge_reference=False
    )
    if annotation_table is None:
        return coda_annotation_columns(neurons.drop(columns=['id']), id_column)

    kinds = client.materialize.get_unique_string_values(annotation_table).get(
        system_column, []
    )
    wide = None
    for kind in kinds:
        rows = client.materialize.query_table(
            annotation_table,
            filter_equal_dict={system_column: kind},
            select_columns=[ref_column, value_column],
            merge_reference=False,
        )
        # First row wins a repeat, as Coda's pivot does: an annotation base can carry two
        # rows for one neuron, and a cross product here would double every downstream count.
        rows = rows.drop_duplicates(subset=[ref_column], keep='first')
        rows = rows.rename(columns={value_column: kind})
        wide = rows if wide is None else wide.merge(rows, on=ref_column, how='outer')

    if wide is None:
        return coda_annotation_columns(neurons.drop(columns=['id']), id_column)
    out = neurons.merge(wide, left_on='id', right_on=ref_column, how='left')
    out = out.drop(columns=[c for c in ('id', ref_column) if c in out.columns])
    return coda_annotation_columns(out, id_column)


class CodaCaveDataset:
    """A CAVE datastack: the client, and the neuron table Coda labels it with.

    `client` is a `caveclient.CAVEclient` pinned to one materialization, so every query
    through it answers from the same frozen snapshot and `client.timestamp` is that
    snapshot's instant.

    `labels` is one row per neuron with Coda's column names — `neuronId`, `type` — built
    from the datastack's own tables, or handed over ready-made when something is wired to
    the Dataset's Annotations socket on the canvas. It is fetched on first use: a graph
    that never asks about neurons should not pay for the index.
    """

    def __init__(
        self,
        client,
        neuron_table=None,
        id_column='pt_root_id',
        annotation_table=None,
        ref_column=None,
        system_column=None,
        value_column=None,
        labels=None,
    ):
        self.client = client
        self.neuron_table = neuron_table
        self.id_column = id_column
        self.annotation_table = annotation_table
        self.ref_column = ref_column
        self.system_column = system_column
        self.value_column = value_column
        self._labels = labels

    @property
    def labels(self):
        if self._labels is None:
            if self.neuron_table is None:
                raise ValueError(
                    'This datastack publishes no neuron table, so the only list of its '
                    'neurons is an annotation source. Wire one to the Dataset on the canvas '
                    'and export again, or pass labels= here.'
                )
            self._labels = coda_cave_neurons(
                self.client,
                self.neuron_table,
                id_column=self.id_column,
                annotation_table=self.annotation_table,
                ref_column=self.ref_column,
                system_column=self.system_column,
                value_column=self.value_column,
            )
        return self._labels


def coda_cave_table(
    client,
    table,
    id_column='pt_root_id',
    columns=None,
    pivot_on=None,
    value_column=None,
):
    """A CAVE annotation table as a Coda neuron table."""
    if pivot_on:
        kinds = client.materialize.get_unique_string_values(table).get(pivot_on, [])
        wide = None
        for kind in kinds:
            rows = client.materialize.query_table(
                table,
                filter_equal_dict={pivot_on: kind},
                select_columns=[id_column, value_column],
                merge_reference=False,
            )
            rows = rows.drop_duplicates(subset=[id_column], keep='first')
            rows = rows.rename(columns={value_column: kind})
            wide = rows if wide is None else wide.merge(rows, on=id_column, how='outer')
        out = wide if wide is not None else pd.DataFrame({id_column: []})
    else:
        select = [id_column] + list(columns) if columns else None
        out = client.materialize.query_table(
            table, select_columns=select, merge_reference=False
        )
        if columns:
            out = out[[id_column] + [c for c in columns if c in out.columns]]
    return coda_annotation_columns(out, id_column)


def coda_cave_table_info(client, table):
    """What one table or view of a datastack is. Prints its facts, returns its columns.

    Four reads: the listing (so a mistyped name can name the alternatives), the
    metadata record, the two row counts, and one real row to read the materialized
    column set off.
    """
    views = client.materialize.get_views()
    tables = client.materialize.get_tables()
    if table in views:
        kind = 'view'
    elif table in tables:
        kind = 'table'
    else:
        raise ValueError(
            f'"{table}" is not a table or view in this datastack. '
            f'Available: {", ".join(sorted(tables) + sorted(views))}'
        )

    if kind == 'table':
        meta = client.materialize.get_table_metadata(table)
        print(f'{table}  ({meta.get("schema_type", "?")})')
        # Two counts, both true, and they disagree by up to a third. The annotation
        # service counts the table as it stands; the materialization engine counts what
        # this snapshot froze. The live one is what predicts a truncated query.
        live = client.annotation.get_annotation_count(table)
        frozen = client.materialize.get_annotation_count(table)
        print(f'  rows: {live:,} live, {frozen:,} in v{client.materialize.version}')
        if meta.get('reference_table'):
            print(f'  annotates: {meta["reference_table"]}')
        # The publisher went out of its way to attach this; every table probed has none.
        if meta.get('notice_text'):
            print(f'  NOTICE: {meta["notice_text"]}')
    else:
        meta = client.materialize.get_view_metadata(table)
        print(f'{table}  (view)')
        print(
            '  note: CAVE does not push a row limit into a view, so an aggregating one'
            ' builds its whole result before handing back the single row read below.'
        )
    if meta.get('description'):
        print()
        print(meta['description'].strip())

    # split_positions=True is what makes these the columns Coda lists: caveclient folds a
    # bound point back into one object column by default, where the app asks for x/y/z.
    query = client.materialize.query_view if kind == 'view' else client.materialize.query_table
    sample = query(table, limit=1, split_positions=True)

    def example(name):
        if sample.empty:
            return ''
        value = sample[name].iloc[0]
        try:
            if pd.isna(value):
                return ''
        except (TypeError, ValueError):
            # An array-valued cell, which pd.isna answers elementwise for.
            pass
        return str(value)

    return pd.DataFrame(
        {
            'column': list(sample.columns),
            'type': [str(dtype) for dtype in sample.dtypes],
            'example': [example(name) for name in sample.columns],
        },
        columns=['column', 'type', 'example'],
    )


def coda_cave_tables(client, include_views=True):
    """Every annotation table in a materialization, and optionally its views.

    Two endpoints rather than one. `get_tables` answers the annotation tables;
    `get_views` answers the saved queries, which is where a datastack's aggregations
    live — FlyWire's connectivity is `valid_connection_v2`, a view, and no table
    holds it.
    """
    rows = [
        {'table': name, 'kind': 'table'}
        for name in sorted(client.materialize.get_tables())
    ]
    if include_views:
        # caveclient 8.2.1 annotates get_views as list[str] and returns a dict keyed by
        # name. sorted() reads the keys either way.
        rows += [
            {'table': name, 'kind': 'view'}
            for name in sorted(client.materialize.get_views())
        ]
    return pd.DataFrame(rows, columns=['table', 'kind'])


def coda_join_annotations(left, right):
    """Chain two annotation sources: outer join on `neuronId`, the later one winning."""
    if left is None:
        return right
    if right is None:
        return left
    left = left.drop_duplicates(subset=['neuronId'], keep='first')
    right = right.drop_duplicates(subset=['neuronId'], keep='first')
    shared = [c for c in right.columns if c in left.columns and c != 'neuronId']
    merged = left.merge(
        right, on='neuronId', how='outer', suffixes=('', '_coda_later')
    )
    for name in shared:
        later = merged[name + '_coda_later']
        # Later wins, falling back to the earlier source where the later one is null.
        merged[name] = later.combine_first(merged[name])
        merged = merged.drop(columns=[name + '_coda_later'])
    return merged


import re

_CODA_OPERATORS = [("==", "eq"), ("!=", "ne"), (">=", "ge"), ("<=", "le"),
                   ("~", "match"), (">", "gt"), ("<", "lt"), ("=", "eq")]
_CODA_FIELD_NAME = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")


def _coda_tokenize(text):
    """Whitespace-split, but quotes hold a token together."""
    tokens, i = [], 0
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break
        start, quote = i, None
        while i < len(text):
            ch = text[i]
            if quote is not None:
                if ch == quote:
                    quote = None
            elif ch in "\"'":
                quote = ch
            elif ch.isspace():
                break
            i += 1
        tokens.append(text[start:i])
    return tokens


def _coda_unquote(value):
    if value[:1] in ("\"", "'") and len(value) >= 2:
        return value[1:-1] if value.endswith(value[0]) else value[1:]
    return value


def _coda_split_operator(token):
    """Field/operator/value, or None for a bare word.

    Operators are tried longest-first so "!=" is not read as "=", and the field has to
    look like a name -- otherwise "LC4-a" would parse as a comparison.
    """
    for symbol, op in _CODA_OPERATORS:
        at = token.find(symbol)
        if at <= 0:
            continue
        field = token[:at]
        if not _CODA_FIELD_NAME.match(field):
            continue
        return field, op, token[at + len(symbol):]
    return None


def _coda_parse_search(text):
    terms = []
    for raw in _coda_tokenize(text):
        negate = False
        if raw[:1] in ("!", "-") and len(raw) > 1 and _coda_split_operator(raw) is None:
            negate, raw = True, raw[1:]
        split = _coda_split_operator(raw)
        if split is None:
            value = _coda_unquote(raw)
            if value:
                terms.append(("text", value.lower(), None, None, negate))
            continue
        field, op, value = split
        value = _coda_unquote(value)
        if not value:
            # Every query mid-typing looks like this; it narrows nothing rather than
            # being an error.
            continue
        terms.append(("field", value, field, op, negate))
    return terms


def _coda_haystack(df):
    """Lowercase text of every searchable column, one string per row.

    String columns and neuronId only -- so a bare "1200" finds a neuron id and does not
    also match every neuron with 1200 synapses.
    """
    cols = [c for c in df.columns
            if df[c].dtype == object or str(c) == "neuronId"]
    if not cols:
        return pd.Series([""] * len(df), index=df.index)
    parts = [df[c].fillna("").astype(str) for c in cols]
    joined = parts[0]
    for part in parts[1:]:
        joined = joined.str.cat(part, sep=" ")
    return joined.str.lower()


def _coda_field_mask(df, field, op, value):
    """One field comparison.

    A missing value satisfies "!=" and nothing else -- so status!=Traced returns the
    untraced *and* the unlabelled, which is the question somebody auditing a dataset
    for gaps is actually asking. SQL's three-valued logic drops both, silently.
    """
    col = next((c for c in df.columns if str(c).lower() == field.lower()), None)
    if col is None:
        return pd.Series(False, index=df.index)
    series = df[col]
    missing = series.isna()

    if op == "match":
        # Unanchored, deliberately unlike neuPrint's "=~": this search is local and has
        # no server semantic to match.
        rx = re.compile(value)
        found = series.fillna("").astype(str).map(lambda v: rx.search(v) is not None)
        return found & ~missing

    if pd.api.types.is_numeric_dtype(series):
        try:
            right = float(value)
        except ValueError:
            return pd.Series(False, index=df.index)
        left = pd.to_numeric(series, errors="coerce")
    else:
        right = value.lower()
        left = series.fillna("").astype(str).str.lower()

    if op == "eq":
        mask = left == right
    elif op == "ne":
        mask = left != right
    elif op == "gt":
        mask = left > right
    elif op == "lt":
        mask = left < right
    elif op == "ge":
        mask = left >= right
    else:
        mask = left <= right

    mask = mask.fillna(False).astype(bool)
    return (mask | missing) if op == "ne" else (mask & ~missing)


def coda_search(df, query):
    """Rows matching Coda's Explore Dataset query language.

    Terms are AND-ed; a leading "!" or "-" negates one. A bare word is a substring of
    the row's searchable text; "field=value" compares one column, with ">" "<" ">=",
    "<=", "!=" and "~" (unanchored regex) as the other operators.

    Two things this does NOT reproduce, both of which change which rows you get:

    * Hits come back in table order. Coda ranks them by relevance, which only matters
      where the result is capped -- but there it decides which rows survive the cap.
    * A query matching nothing returns nothing. Coda retries it as a subsequence, so
      "mechnosensory" still finds "mechanosensory" there and finds nothing here.
    """
    terms = _coda_parse_search(query)
    if not terms:
        return df

    keep = pd.Series(True, index=df.index)
    haystack = None
    for kind, value, field, op, negate in terms:
        if kind == "text":
            if haystack is None:
                haystack = _coda_haystack(df)
            mask = haystack.str.contains(value, regex=False)
        else:
            mask = _coda_field_mask(df, field, op, value)
        keep &= ~mask if negate else mask

    return df[keep]


def coda_seatable(table, id_column='root_id', columns=None):
    """A SeaTable table as a Coda neuron table."""
    df = table.to_frame(row_id_index=False)
    # sea-serpent names its columns with numpy `str_`, which indexes fine and reads oddly
    # in anything that prints the column list.
    df.columns = [str(c) for c in df.columns]
    if columns:
        keep = [c for c in columns if c in df.columns and c != id_column]
    else:
        # Empty means every column but the id, which is what a base says without being
        # asked what "every" is.
        keep = [c for c in df.columns if c != id_column]
    return coda_annotation_columns(df[[id_column] + keep], id_column)


def coda_update_root_ids(
    client, df, id_column='neuronId', supervoxel_column='supervoxel_id'
):
    """Repair root ids that were retired before this materialization was frozen."""
    out = df.copy()
    ids = coda_int64(out[id_column])
    svids = coda_int64(out[supervoxel_column])
    askable = ids > 0
    if not askable.any():
        return out

    # Current at the materialization? Only the rows that are not get looked up.
    latest = client.chunkedgraph.is_latest_roots(
        ids[askable].astype('int64').to_numpy(), timestamp=client.timestamp
    )
    stale = pd.Series(False, index=out.index)
    stale.loc[askable] = ~np.asarray(latest, dtype=bool)
    stale &= svids > 0
    if not stale.any():
        return out

    roots = client.chunkedgraph.get_roots(
        svids[stale].astype('int64').to_numpy(), timestamp=client.timestamp
    )
    repaired = pd.Series(np.asarray(roots), index=out.index[stale])
    # A supervoxel the graph does not know answers 0, which is not a root to write anywhere.
    repaired = repaired[repaired > 0]
    if repaired.empty:
        return out
    if out[id_column].dtype == object:
        repaired = repaired.astype(str)
    else:
        # Keep the column's own storage rather than widening it to uint64 or object,
        # which would change how every later comparison and sort behaves.
        repaired = repaired.astype(out[id_column].dtype)
    out.loc[repaired.index, id_column] = repaired
    return out

In [ ]:
# ── SeaTable ──
_sea = ss.Table(
    'types',
    base='my base',
    server='https://cloud.seatable.io',
    auth_token=os.environ['SEATABLE_TOKEN'],
)
seatable = coda_seatable(
    _sea,
    id_column='root_id',
)

In [ ]:
# ── Custom CAVE ──
custom_cave = CodaCaveDataset(
    CAVEclient('wclee_aedes_brain', version=117),
    neuron_table='nuclei',
    id_column='pt_root_id',
)

In [ ]:
# ── List CAVE tables ──
# NOTE: The Dataset wired here is a reference — it names a datastack rather
# than taking its value — and its cell is written below this one, so this
# builds its own client for the same datastack and materialization.
_cave = CAVEclient('flywire_fafb_public', version=783)
list_cave_tables = coda_cave_tables(_cave, include_views=True)

In [ ]:
# ── CAVE table info ──
# NOTE: The card in Coda shows this table’s description and row counts; here
# they are printed, and the column listing is what the cell binds.
_cave = CAVEclient('flywire_fafb_public', version=783)
cave_table_info = coda_cave_table_info(_cave, 'nuclei_v1')

In [ ]:
# ── FlyTable ──
_sea = ss.Table(
    'info',
    base='main',
    server='https://flytable.mrc-lmb.cam.ac.uk',
    auth_token=os.environ['SEATABLE_TOKEN'],
)
# Every column is downloaded and then narrowed, as it is on the canvas. To narrow it
# server-side instead — measured at about 4x faster, at the cost of sea-serpent's
# dtype conversion — replace the call below with:
#     _rows = _sea.query('SELECT `root_id`, `cell_type`, `side` FROM `info`', no_limit=True)
#     flytable = coda_annotation_columns(pd.DataFrame(_rows), 'root_id')
flytable = coda_seatable(
    _sea,
    id_column='root_id',
    columns=['cell_type', 'side'],
)
flytable = coda_join_annotations(seatable, flytable)

In [ ]:
# ── CAVE table ──
_cave = CAVEclient('flywire_fafb_public', version=783)
cave_table = coda_cave_table(
    _cave,
    'nuclei_v1',
    id_column='pt_root_id',
    columns=['volume'],
)
cave_table = coda_join_annotations(flytable, cave_table)

In [ ]:
# ── CAVE table ──
# NOTE: The Dataset wired here is a reference — it names a datastack rather
# than taking its value — and its cell is written below this one, so this
# builds its own client for the same datastack and materialization.
_cave = CAVEclient('flywire_fafb_public', version=783)
cave_table_2 = coda_cave_table(
    _cave,
    'hierarchical_neuron_annotations',
    id_column='target_id',
    pivot_on='classification_system',
    value_column='cell_type',
)
cave_table_2 = coda_join_annotations(cave_table, cave_table_2)

In [ ]:
# ── Filter ──
filter_ = cave_table_2[cave_table_2['type'].notna() & (cave_table_2['type'] != '')]

In [ ]:
# ── Update root IDs ──
# NOTE: The Dataset wired here is a reference — it names a datastack rather
# than taking its value — and its cell is written below this one, so this
# builds its own client for the same datastack and materialization.
_repair_at = CAVEclient('flywire_fafb_public', version=783)
update_root_ids = coda_update_root_ids(
    _repair_at,
    filter_,
    id_column='neuronId',
    supervoxel_column='supervoxel_id',
)

In [ ]:
# ── FlyWire FAFB (CAVE) ──
# NOTE: Annotations are wired to this dataset on the canvas, so they replace
# the datastack's own labels rather than adding to them.
flywire_fafb_cave = CodaCaveDataset(
    CAVEclient('flywire_fafb_public', version=783),
    labels=update_root_ids,
)

In [ ]:
# ── Table ──
table_out = update_root_ids
table_filtered = table_out
table_out

In [ ]:
# ── Find Neurons ──
# NOTE: A CAVE datastack has no server-side neuron query, so Coda reads its
# whole index once and filters it here — which is what this does. Explore
# Dataset shares the same frame.
find_neurons = flywire_fafb_cave.labels
find_neurons = find_neurons[find_neurons['type'].astype(str).str.contains('^(?:LC.*)$', regex=True, case=True, na=False)]

In [ ]:
# ── Dataset Summary ──
# TODO: "Dataset Summary" is wired to a CAVE dataset, and its notebook cell
# has only been written for neuPrint. The dataset itself is a real client,
# so this is the step to fill in by hand.

In [ ]:
# ── Explore Dataset ──
# NOTE: Explore Dataset searches the whole neuron table locally. This is the
# datastack’s own index — its neuron table joined to its annotations, or
# whatever is wired to the Dataset’s Annotations socket — fetched the first
# time anything asks for it. On FlyWire that is 139,255 rows and takes a few
# seconds.
explore_dataset_all_ = flywire_fafb_cave.labels

explore_dataset_hits = coda_search(explore_dataset_all_, 'side=left')

_selected_ids = [720575940628857210]
explore_dataset_selected = explore_dataset_all_[explore_dataset_all_['neuronId'].astype(str).isin([str(_i) for _i in _selected_ids])]